# HAM10000 — Clasificación con entrenamiento en dos etapas (fine-tuning)

Estrategia: entrenar EfficientNet-B0 en dos fases para reducir el *distributional shift* de las imágenes sintéticas.

| Etapa | Datos | Epochs | LR |
|-------|-------|--------|-----|
| Stage 1 | real_mel + sintéticas | 12 | 1e-4 |
| Stage 2 (fine-tune) | solo reales | 3 | 1e-5 |

| Escenario | S1 train mel | Hipótesis |
|-----------|-------------|----------|
| `lora_2x_ft` | 801 real + 801 LoRA | ¿FT corrige el shift de LoRA (lora_2x AUC=0.9211)? |
| `lora_3x_ft` | 801 real + 1602 LoRA | ¿3x LoRA + FT mejora sobre 2x? |
| `gan_final_2x_ft` | 801 real + 801 GAN | ¿FT mejora sobre gan_final_2x (AUC=0.9323)? |
| `gan_final_3x_ft` | 801 real + 1602 GAN | ¿3x + FT supera el SOTA actual (gan64_2x=0.9378)? |
| `derm_s040_2x_ft` | 801 real + 801 Derm-T2IM (s=0.40) | ¿Derm-T2IM (mismo dominio) reduce el shift? |
| `derm_s005_2x_ft` | 801 real + 801 Derm-T2IM (s=0.05) | ¿Perturbaciones sutiles + FT actúan como regularización? |

**Prerequisitos Drive:** `classification_data.zip`, `synthetic/lora/`, `gan_final.zip`, `synthetic/derm_s040/`, `synthetic/derm_s005/`

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import torch

def get_device():
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'CUDA: {name}  ({vram:.1f} GB)')
        return torch.device('cuda'), vram
    if torch.backends.mps.is_available():
        print('Apple MPS')
        return torch.device('mps'), 0
    print('CPU')
    return torch.device('cpu'), 0

DEVICE, VRAM_GB = get_device()
print(f'IN_COLAB={IN_COLAB}  device={DEVICE}')

In [ ]:
from pathlib import Path
import zipfile, shutil, json, subprocess, sys

if IN_COLAB:
    drive.mount('/content/drive')
    DRIVE_ROOT    = Path('/content/drive/MyDrive/ham10000-augmentation')
    ZIP_PATH      = DRIVE_ROOT / 'classification_data.zip'
    IMAGES_DIR    = Path('/content/images')
    SPLITS_DIR    = Path('/content/splits')
    SYNTH_ROOT    = DRIVE_ROOT / 'synthetic'
    GAN_FINAL_ZIP = next(DRIVE_ROOT.glob('gan_final*.zip'), None)
    GAN_FINAL_DIR = Path('/content/gan_final')
    EXP_ROOT      = DRIVE_ROOT / 'experiments'
else:
    PROJECT_ROOT  = Path.cwd()
    ZIP_PATH      = PROJECT_ROOT / 'data/processed/classification_data.zip'
    IMAGES_DIR    = PROJECT_ROOT / 'data/processed/images'
    SPLITS_DIR    = PROJECT_ROOT / 'data/processed/splits'
    SYNTH_ROOT    = PROJECT_ROOT / 'data/synthetic'
    GAN_FINAL_ZIP = next(PROJECT_ROOT.glob('gan_final*.zip'), None)
    GAN_FINAL_DIR = PROJECT_ROOT / 'data/synthetic/gan_final'
    EXP_ROOT      = PROJECT_ROOT / 'experiments'

LORA_DIR      = SYNTH_ROOT / 'lora'
DERM_S040_DIR = SYNTH_ROOT / 'derm_s040'
DERM_S005_DIR = SYNTH_ROOT / 'derm_s005'
EXP_ROOT.mkdir(parents=True, exist_ok=True)

print(f'LORA_DIR:      {LORA_DIR}')
print(f'GAN_FINAL_DIR: {GAN_FINAL_DIR}')
print(f'DERM_S040_DIR: {DERM_S040_DIR}')
print(f'DERM_S005_DIR: {DERM_S005_DIR}')
print(f'EXP_ROOT:      {EXP_ROOT}')

In [ ]:
def pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

try:
    import timm; print(f'timm {timm.__version__}')
except ImportError:
    pip('timm'); import timm
try:
    import sklearn; print(f'sklearn {sklearn.__version__}')
except ImportError:
    pip('scikit-learn'); import sklearn

print('Dependencias listas')

In [ ]:
# ── Estado rápido — correr sin cargar modelos ─────────────────────────────────
FT_SCENARIOS  = ['lora_2x_ft', 'lora_3x_ft', 'gan_final_2x_ft', 'gan_final_3x_ft',
                 'derm_s040_2x_ft', 'derm_s005_2x_ft']
REF_SCENARIOS = ['real_only', 'gan_final_2x', 'lora_2x', 'gan64_2x']

def find_run_dir(sc):
    pointer = EXP_ROOT / f'{sc}_current.txt'
    if pointer.exists():
        return EXP_ROOT / pointer.read_text().strip()
    completed = sorted([d for d in EXP_ROOT.glob(f'*_{sc}')
                        if (d / 'test_metrics.json').exists()])
    return completed[-1] if completed else None

print('── Escenarios de fine-tuning ──')
for sc in FT_SCENARIOS:
    run_dir = find_run_dir(sc)
    if run_dir and (run_dir / 'test_metrics.json').exists():
        m = json.loads((run_dir / 'test_metrics.json').read_text())
        print(f'  ✅ {sc:24s}  AUC={m["auc"]}  Recall={m["recall_mel"]}  F1={m["f1_mel"]}')
    elif run_dir:
        s1 = (run_dir / 'stage1_done.pt').exists()
        ck = (run_dir / 'checkpoint_last.pt').exists()
        estado = 'S2-retomable' if (s1 and ck) else ('S1-retomable' if ck else 'iniciado')
        print(f'  🔄 {sc:24s}  {estado}')
    else:
        print(f'  ⬜ {sc}')

print('\n── Referencia (previos) ──')
for sc in REF_SCENARIOS:
    run_dir = find_run_dir(sc)
    if run_dir and (run_dir / 'test_metrics.json').exists():
        m = json.loads((run_dir / 'test_metrics.json').read_text())
        print(f'  ✅ {sc:24s}  AUC={m["auc"]}  Recall={m["recall_mel"]}  F1={m["f1_mel"]}')
    else:
        print(f'  ⬜ {sc}  (no encontrado)')

In [ ]:
# ── Extraer imágenes reales + splits ─────────────────────────────────────────
if IN_COLAB:
    IMAGES_DIR.mkdir(parents=True, exist_ok=True)
    SPLITS_DIR.mkdir(parents=True, exist_ok=True)
    if len(list(IMAGES_DIR.glob('*.jpg'))) < 100:
        print('Extrayendo classification_data.zip...')
        with zipfile.ZipFile(ZIP_PATH) as zf:
            for m in zf.infolist():
                data = zf.read(m.filename)
                if m.filename.startswith('images/') and m.filename.endswith('.jpg'):
                    (IMAGES_DIR / Path(m.filename).name).write_bytes(data)
                elif m.filename.startswith('splits/') and m.filename.endswith('.csv'):
                    (SPLITS_DIR / Path(m.filename).name).write_bytes(data)
        print(f'  {len(list(IMAGES_DIR.glob("*.jpg")))} imágenes  |  splits listos')
    else:
        print(f'Imágenes ya extraídas ({len(list(IMAGES_DIR.glob("*.jpg")))})')

In [ ]:
# ── Copiar sintéticas de Drive → /content/; extraer GAN-final ─────────────────
if IN_COLAB:
    LOCAL_SYNTH = Path('/content/synthetic')
    if not LOCAL_SYNTH.exists():
        print('Copiando synthetic/ de Drive...')
        shutil.copytree(str(SYNTH_ROOT), str(LOCAL_SYNTH))
        print(f'  {len(list(LOCAL_SYNTH.glob("**/*.jpg")))} imágenes copiadas')
    else:
        print('Sintéticas ya en local')
    SYNTH_ROOT    = LOCAL_SYNTH
    LORA_DIR      = SYNTH_ROOT / 'lora'
    DERM_S040_DIR = SYNTH_ROOT / 'derm_s040'
    DERM_S005_DIR = SYNTH_ROOT / 'derm_s005'

    GAN_FINAL_DIR.mkdir(parents=True, exist_ok=True)
    if len(list(GAN_FINAL_DIR.glob('*.png'))) < 10:
        assert GAN_FINAL_ZIP and GAN_FINAL_ZIP.exists(), 'Sube gan_final.zip a Drive'
        print(f'Extrayendo {GAN_FINAL_ZIP.name}...')
        with zipfile.ZipFile(GAN_FINAL_ZIP) as zf:
            for m in zf.infolist():
                if m.filename.endswith('.png'):
                    (GAN_FINAL_DIR / Path(m.filename).name).write_bytes(zf.read(m.filename))

n_lora = len(list(LORA_DIR.glob('*.jpg')))          if LORA_DIR.exists()      else 0
n_gan  = len(list(GAN_FINAL_DIR.glob('*.png')))     if GAN_FINAL_DIR.exists() else 0
n_d040 = len(list(DERM_S040_DIR.glob('*.jpg')))     if DERM_S040_DIR.exists() else 0
n_d005 = len(list(DERM_S005_DIR.glob('*.jpg')))     if DERM_S005_DIR.exists() else 0
print(f'LoRA: {n_lora}  |  GAN-final: {n_gan}  |  Derm-s040: {n_d040}  |  Derm-s005: {n_d005}')

In [ ]:
# ── Hiperparámetros ───────────────────────────────────────────────────────────
import random, numpy as np

S1_EPOCHS = 12
S2_EPOCHS = 3
S1_LR     = 1e-4
S2_LR     = 1e-5
SEED      = 42

if DEVICE.type == 'cuda':
    BATCH_SIZE  = 64 if VRAM_GB >= 40 else 32
    NUM_WORKERS = 2
elif DEVICE.type == 'mps':
    BATCH_SIZE, NUM_WORKERS = 16, 0
else:
    BATCH_SIZE, NUM_WORKERS = 8, 0

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if DEVICE.type == 'cuda': torch.cuda.manual_seed_all(SEED)

print(f'S1={S1_EPOCHS}ep lr={S1_LR}  |  S2={S2_EPOCHS}ep lr={S2_LR}  |  BATCH={BATCH_SIZE}')

In [ ]:
# ── Dataset y utilidades ──────────────────────────────────────────────────────
import pandas as pd, time
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

TRAIN_TF = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
EVAL_TF = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

class FlatDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples; self.transform = transform
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label
    def class_weights(self):
        labels = np.array([l for _,l in self.samples])
        counts = np.bincount(labels)
        w = 1.0 / counts.astype(float)
        return torch.tensor([w[l] for _,l in self.samples], dtype=torch.float)

def make_loader(samples, transform, weighted=False):
    ds  = FlatDataset(samples, transform)
    pin = DEVICE.type == 'cuda'
    if weighted:
        sampler = WeightedRandomSampler(ds.class_weights(), len(ds), replacement=True)
        return DataLoader(ds, batch_size=BATCH_SIZE, sampler=sampler,
                         num_workers=NUM_WORKERS, pin_memory=pin)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                     num_workers=NUM_WORKERS, pin_memory=pin)

def resolve_path(rel_path):
    if IN_COLAB: return IMAGES_DIR / Path(rel_path).name
    return Path.cwd() / rel_path

def load_splits():
    def read(csv_path, label_filter=None):
        df = pd.read_csv(csv_path)
        if label_filter is not None: df = df[df['label'] == label_filter]
        return [(resolve_path(row['image_path']), int(row['label'])) for _,row in df.iterrows()]
    nv  = read(SPLITS_DIR/'train.csv', 0)
    mel = read(SPLITS_DIR/'train.csv', 1)
    val = read(SPLITS_DIR/'val.csv')
    tst = read(SPLITS_DIR/'test.csv')
    print(f'train  nv:{len(nv)}  mel:{len(mel)} | val:{len(val)} | test:{len(tst)}')
    return nv, mel, val, tst

def synth_samples_from(paths, n, seed=SEED):
    paths = list(paths); random.Random(seed).shuffle(paths)
    return [(p, 1) for p in paths[:n]]

print('Utilidades listas')

In [ ]:
# ── Entrenamiento en dos etapas con resume ────────────────────────────────────
import timm, torch.nn as nn
from datetime import datetime, timezone
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm
from sklearn.metrics import (f1_score, roc_auc_score, recall_score,
    classification_report, confusion_matrix, roc_curve, ConfusionMatrixDisplay)
import matplotlib.pyplot as plt

try:
    import numpy as _np
    torch.serialization.add_safe_globals([_np._core.multiarray.scalar])
except AttributeError:
    pass

def build_model():
    return timm.create_model('efficientnet_b0', pretrained=True, num_classes=2)

def _eval_loop(model, loader, criterion):
    model.eval()
    loss_sum, total = 0.0, 0
    preds, labels, probs = [], [], []
    with torch.no_grad():
        for imgs, lbs in loader:
            imgs, lbs = imgs.to(DEVICE), lbs.to(DEVICE)
            logits = model(imgs)
            loss   = criterion(logits, lbs)
            p      = torch.softmax(logits, 1)[:, 1]
            loss_sum += loss.item() * len(lbs); total += len(lbs)
            preds.extend(logits.argmax(1).cpu().tolist())
            labels.extend(lbs.cpu().tolist())
            probs.extend(p.cpu().tolist())
    return {'loss':       round(loss_sum / total, 4),
            'f1_mel':     round(f1_score(labels, preds, pos_label=1, zero_division=0), 4),
            'recall_mel': round(recall_score(labels, preds, pos_label=1, zero_division=0), 4),
            'auc':        round(roc_auc_score(labels, probs), 4)}


def _train_stage(name, stage, model, loader, val_loader, criterion,
                 optimizer, scheduler, n_epochs, history, best_f1,
                 run_dir, ckpt_path, start_epoch=1):
    """Ejecuta un stage de entrenamiento; retorna (best_f1, history)."""
    for epoch in range(start_epoch, n_epochs + 1):
        t0 = time.time()
        model.train()
        tr_loss, correct, total = 0.0, 0, 0
        pbar = tqdm(loader, desc=f'[{name}] S{stage} {epoch:02d}/{n_epochs}', leave=False)
        for imgs, lbs in pbar:
            imgs, lbs = imgs.to(DEVICE), lbs.to(DEVICE)
            optimizer.zero_grad()
            logits = model(imgs); loss = criterion(logits, lbs)
            loss.backward(); optimizer.step()
            tr_loss += loss.item() * len(lbs)
            correct += (logits.argmax(1) == lbs).sum().item()
            total   += len(lbs)
            pbar.set_postfix(loss=f'{loss.item():.4f}', acc=f'{correct/total:.3f}')
        scheduler.step()
        val_m   = _eval_loop(model, val_loader, criterion)
        elapsed = time.time() - t0
        row = {'stage': stage, 'epoch': epoch,
               'train_loss': round(tr_loss / total, 4),
               'train_acc':  round(correct / total, 4),
               **{f'val_{k}': v for k, v in val_m.items()},
               'elapsed_s':  round(elapsed, 1)}
        history.append(row)
        print(f'  S{stage} {epoch:02d}/{n_epochs} | loss={tr_loss/total:.4f} '
              f'| val_f1={val_m["f1_mel"]:.3f} val_auc={val_m["auc"]:.3f} | {elapsed:.0f}s')
        if val_m['f1_mel'] >= best_f1:
            best_f1 = val_m['f1_mel']
            torch.save(model.state_dict(), run_dir / 'best_model.pt')
        torch.save({'stage': stage, 'epoch': epoch,
                    'model': model.state_dict(),
                    'optimizer': optimizer.state_dict(),
                    'scheduler': scheduler.state_dict(),
                    'best_f1': best_f1, 'history': history}, ckpt_path)
    return best_f1, history


def run_finetune(name, stage1_samples, stage2_samples, val_samples, test_samples):
    """
    Entrenamiento en dos etapas:
      Stage 1: stage1_samples (real + sintéticas)  — S1_EPOCHS epochs, lr=S1_LR
      Stage 2: stage2_samples (solo reales)         — S2_EPOCHS epochs, lr=S2_LR
    Resume automático si se cae la sesión.
    """
    pointer     = EXP_ROOT / f'{name}_current.txt'
    run_id      = (pointer.read_text().strip() if pointer.exists()
                   else f'{datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")}_{name}')
    if not pointer.exists(): pointer.write_text(run_id)

    run_dir     = EXP_ROOT / run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    done_marker = run_dir / 'test_metrics.json'
    ckpt_path   = run_dir / 'checkpoint_last.pt'
    s1_done     = run_dir / 'stage1_done.pt'

    if done_marker.exists():
        metrics = json.loads(done_marker.read_text())
        pointer.unlink(missing_ok=True)
        print(f'[{name}] Ya completado — AUC={metrics["auc"]}  '
              f'Recall={metrics["recall_mel"]}  F1={metrics["f1_mel"]}')
        return metrics

    n_nv  = sum(1 for _, l in stage1_samples if l == 0)
    n_mel = sum(1 for _, l in stage1_samples if l == 1)
    print(f'\n{"="*60}\nEscenario: {name}')
    print(f'S1 train  nv={n_nv}  mel={n_mel}  |  '
          f'S2 real_only nv+mel={len(stage2_samples)}  |  val={len(val_samples)}')
    print(f'{"="*60}')

    if not (run_dir / 'config.json').exists():
        (run_dir / 'config.json').write_text(json.dumps({
            'run_id': run_id, 'scenario': name, 'model': 'efficientnet_b0',
            'pretrained': True, 's1_epochs': S1_EPOCHS, 's2_epochs': S2_EPOCHS,
            'batch_size': BATCH_SIZE, 's1_lr': S1_LR, 's2_lr': S2_LR, 'seed': SEED,
            's1_train_nv': n_nv, 's1_train_mel': n_mel,
            'train_mel': n_mel,  # para comparación con otros escenarios
            'started_at': datetime.now(timezone.utc).isoformat()}, indent=2))

    criterion = nn.CrossEntropyLoss()
    val_loader  = make_loader(val_samples,  EVAL_TF)
    test_loader = make_loader(test_samples, EVAL_TF)

    model   = build_model().to(DEVICE)
    history = []
    best_f1 = 0.0

    # ── Stage 1 ───────────────────────────────────────────────────────────────
    if not s1_done.exists():
        print(f'\n[Stage 1]  {S1_EPOCHS} epochs, lr={S1_LR}')
        train_loader_s1 = make_loader(stage1_samples, TRAIN_TF, weighted=True)
        optimizer  = AdamW(model.parameters(), lr=S1_LR, weight_decay=1e-4)
        scheduler  = CosineAnnealingLR(optimizer, T_max=S1_EPOCHS)
        start_ep   = 1

        if ckpt_path.exists():
            ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
            if ckpt.get('stage') == 1:
                model.load_state_dict(ckpt['model'])
                optimizer.load_state_dict(ckpt['optimizer'])
                scheduler.load_state_dict(ckpt['scheduler'])
                start_ep = ckpt['epoch'] + 1
                best_f1  = ckpt['best_f1']
                history  = ckpt['history']
                print(f'  Retomando Stage 1 desde epoch {start_ep}')

        best_f1, history = _train_stage(
            name, 1, model, train_loader_s1, val_loader, criterion,
            optimizer, scheduler, S1_EPOCHS, history, best_f1,
            run_dir, ckpt_path, start_ep)

        torch.save({'best_f1_s1': best_f1}, s1_done)
        ckpt_path.unlink(missing_ok=True)
        print(f'  Stage 1 completo. best_val_f1={best_f1:.4f}')
    else:
        # Reanudar: reconstruir history desde checkpoint de S2 si existe
        if ckpt_path.exists():
            ckpt    = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
            history = ckpt.get('history', [])
            best_f1 = ckpt.get('best_f1', 0.0)
        print(f'  Stage 1 ya completado — pasando a Stage 2')

    # ── Stage 2 (fine-tune en real only) ──────────────────────────────────────
    print(f'\n[Stage 2]  {S2_EPOCHS} epochs, lr={S2_LR}  — datos reales únicamente')
    train_loader_s2 = make_loader(stage2_samples, TRAIN_TF, weighted=True)
    optimizer2  = AdamW(model.parameters(), lr=S2_LR, weight_decay=1e-4)
    scheduler2  = CosineAnnealingLR(optimizer2, T_max=S2_EPOCHS)
    start_ep2   = 1

    if ckpt_path.exists():
        ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        if ckpt.get('stage') == 2:
            model.load_state_dict(ckpt['model'])
            optimizer2.load_state_dict(ckpt['optimizer'])
            scheduler2.load_state_dict(ckpt['scheduler'])
            start_ep2 = ckpt['epoch'] + 1
            best_f1   = ckpt['best_f1']
            history   = ckpt['history']
            print(f'  Retomando Stage 2 desde epoch {start_ep2}')
        # Si el ckpt es de S1 (raro pero posible), ignorar — el best_model.pt ya existe
    else:
        # Cargar el mejor modelo de S1 como punto de partida del fine-tune
        model.load_state_dict(torch.load(run_dir / 'best_model.pt',
                                          map_location=DEVICE, weights_only=False))
        print('  Cargando best_model de S1 para fine-tune')

    best_f1, history = _train_stage(
        name, 2, model, train_loader_s2, val_loader, criterion,
        optimizer2, scheduler2, S2_EPOCHS, history, best_f1,
        run_dir, ckpt_path, start_ep2)

    # ── Evaluación final ──────────────────────────────────────────────────────
    (run_dir / 'history.json').write_text(json.dumps(history, indent=2))
    model.load_state_dict(torch.load(run_dir / 'best_model.pt',
                                     map_location=DEVICE, weights_only=False))
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for imgs, lbs in test_loader:
            logits = model(imgs.to(DEVICE)); p = torch.softmax(logits, 1)[:, 1]
            all_preds.extend(logits.argmax(1).cpu().tolist())
            all_labels.extend(lbs.tolist())
            all_probs.extend(p.cpu().tolist())

    report  = classification_report(all_labels, all_preds,
                                     target_names=['nv','mel'], output_dict=True)
    auc     = roc_auc_score(all_labels, all_probs)
    metrics = {'auc':           round(auc, 4),
               'f1_mel':        round(report['mel']['f1-score'], 4),
               'recall_mel':    round(report['mel']['recall'], 4),
               'precision_mel': round(report['mel']['precision'], 4),
               'f1_nv':         round(report['nv']['f1-score'], 4),
               'accuracy':      round(report['accuracy'], 4)}
    done_marker.write_text(json.dumps(metrics, indent=2))

    cfg = json.loads((run_dir / 'config.json').read_text())
    cfg.update({'finished_at': datetime.now(timezone.utc).isoformat(),
                'test_metrics': metrics})
    (run_dir / 'config.json').write_text(json.dumps(cfg, indent=2))

    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(cm, display_labels=['nv','mel']).plot(ax=ax, colorbar=False)
    ax.set_title(f'Confusion Matrix — {name}'); fig.tight_layout()
    fig.savefig(run_dir / 'confusion_matrix.png', dpi=150); plt.close(fig)

    # ROC
    fpr, tpr, _ = roc_curve(all_labels, all_probs)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(fpr, tpr, label=f'AUC={auc:.3f}')
    ax.plot([0,1],[0,1],'k--', lw=0.8)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title(f'ROC — {name}'); ax.legend()
    fig.tight_layout(); fig.savefig(run_dir / 'roc_curve.png', dpi=150); plt.close(fig)

    # Curva de entrenamiento por stage
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for stage_id, color in [(1, '#4C72B0'), (2, '#C44E52')]:
        rows = [r for r in history if r['stage'] == stage_id]
        if not rows: continue
        epochs = [r['epoch'] for r in rows]
        axes[0].plot(epochs, [r['val_auc']    for r in rows],
                     color=color, marker='o', label=f'Stage {stage_id}')
        axes[1].plot(epochs, [r['val_f1_mel'] for r in rows],
                     color=color, marker='o', label=f'Stage {stage_id}')
    for ax, title in zip(axes, ['Val AUC', 'Val F1 Melanoma']):
        ax.set_title(title); ax.legend(); ax.set_xlabel('Epoch')
    fig.suptitle(name); fig.tight_layout()
    fig.savefig(run_dir / 'training_curve.png', dpi=150); plt.close(fig)

    ckpt_path.unlink(missing_ok=True)
    pointer.unlink(missing_ok=True)
    print(f'\n[{name}] COMPLETADO  AUC={metrics["auc"]}  '
          f'Recall={metrics["recall_mel"]}  F1={metrics["f1_mel"]}')
    return metrics


print('Función run_finetune lista')

In [ ]:
# ── Cargar splits y fuentes ───────────────────────────────────────────────────
train_nv, train_mel, val_samples, test_samples = load_splits()

# Datos reales para Stage 2 (sin sintéticos)
real_only_train = train_nv + train_mel

n_2x   = len(train_mel)   # 801
n_3x   = n_2x * 2         # 1602

lora_paths      = sorted(LORA_DIR.glob('*.jpg'))          if LORA_DIR.exists()      else []
gan_paths       = sorted(GAN_FINAL_DIR.glob('*.png'))     if GAN_FINAL_DIR.exists() else []
derm_s040_paths = sorted(DERM_S040_DIR.glob('*.jpg'))     if DERM_S040_DIR.exists() else []
derm_s005_paths = sorted(DERM_S005_DIR.glob('*.jpg'))     if DERM_S005_DIR.exists() else []

print(f'LoRA:      {len(lora_paths)} imágenes  (necesito {n_2x} para 2x, {n_3x} para 3x)')
print(f'GAN-final: {len(gan_paths)} imágenes')
print(f'Derm-s040: {len(derm_s040_paths)} imágenes  (necesito {n_2x} para 2x)')
print(f'Derm-s005: {len(derm_s005_paths)} imágenes  (necesito {n_2x} para 2x)')
print(f'Stage 2 (real only): {len(real_only_train)} muestras')

## Escenario 1 — `lora_2x_ft`
**S1:** 801 real mel + 801 LoRA → **S2:** fine-tune real only

Hipótesis: el fine-tune corrige el shift de distribución que hace que `lora_2x` (AUC=0.9211) rinda peor de lo esperado dado su FID=121.71.

In [ ]:
s1_lora_2x = train_nv + train_mel + synth_samples_from(lora_paths, n=n_2x)

results_lora_2x_ft = run_finetune(
    name           = 'lora_2x_ft',
    stage1_samples = s1_lora_2x,
    stage2_samples = real_only_train,
    val_samples    = val_samples,
    test_samples   = test_samples,
)

## Escenario 2 — `lora_3x_ft`
**S1:** 801 real mel + 1602 LoRA (3x) → **S2:** fine-tune real only

Testa si más cantidad de LoRA (3x) junto con fine-tune mejora sobre `lora_2x_ft`.

In [ ]:
s1_lora_3x = train_nv + train_mel + synth_samples_from(lora_paths, n=n_3x)

results_lora_3x_ft = run_finetune(
    name           = 'lora_3x_ft',
    stage1_samples = s1_lora_3x,
    stage2_samples = real_only_train,
    val_samples    = val_samples,
    test_samples   = test_samples,
)

## Escenario 3 — `gan_final_2x_ft`
**S1:** 801 real mel + 801 GAN-final → **S2:** fine-tune real only

Control directo: compara contra `gan_final_2x` (AUC=0.9323) para medir el efecto del fine-tune aislado.

In [ ]:
s1_gan_2x = train_nv + train_mel + synth_samples_from(gan_paths, n=n_2x)

results_gan_2x_ft = run_finetune(
    name           = 'gan_final_2x_ft',
    stage1_samples = s1_gan_2x,
    stage2_samples = real_only_train,
    val_samples    = val_samples,
    test_samples   = test_samples,
)

## Escenario 4 — `gan_final_3x_ft`
**S1:** 801 real mel + 1602 GAN-final (3x) → **S2:** fine-tune real only

Combina más datos sintéticos con fine-tune. Objetivo: superar `gan64_2x` (AUC=0.9378, actual mejor resultado).

In [ ]:
s1_gan_3x = train_nv + train_mel + synth_samples_from(gan_paths, n=n_3x)

results_gan_3x_ft = run_finetune(
    name           = 'gan_final_3x_ft',
    stage1_samples = s1_gan_3x,
    stage2_samples = real_only_train,
    val_samples    = val_samples,
    test_samples   = test_samples,
)

## Escenario 5 — `derm_s040_2x_ft`
**S1:** 801 real mel + 801 Derm-T2IM (strength=0.40) → **S2:** fine-tune real only

Hipótesis: Derm-T2IM fue entrenado sobre HAM10000 + ISIC (mismo dominio que el test set), por lo que el distributional shift debería ser menor que con LoRA o GAN; el fine-tune debería consolidar esa ventaja.

In [ ]:
s1_derm_s040_2x = train_nv + train_mel + synth_samples_from(derm_s040_paths, n=n_2x)

results_derm_s040_2x_ft = run_finetune(
    name           = 'derm_s040_2x_ft',
    stage1_samples = s1_derm_s040_2x,
    stage2_samples = real_only_train,
    val_samples    = val_samples,
    test_samples   = test_samples,
)

## Escenario 6 — `derm_s005_2x_ft`
**S1:** 801 real mel + 801 Derm-T2IM (strength=0.05) → **S2:** fine-tune real only

Hipótesis: perturbaciones casi imperceptibles (strength=0.05) actúan como augmentation conservadora; el fine-tune preserva las características clínicas exactas aprendidas en Stage 1.

In [ ]:
s1_derm_s005_2x = train_nv + train_mel + synth_samples_from(derm_s005_paths, n=n_2x)

results_derm_s005_2x_ft = run_finetune(
    name           = 'derm_s005_2x_ft',
    stage1_samples = s1_derm_s005_2x,
    stage2_samples = real_only_train,
    val_samples    = val_samples,
    test_samples   = test_samples,
)

## Comparación — fine-tuning vs escenarios previos

In [ ]:
# Escenarios de referencia + nuevos de fine-tuning
COMPARE_SCENARIOS = [
    'real_only', 'gan64_2x', 'gan_final_2x', 'lora_2x',
    'lora_2x_ft', 'lora_3x_ft', 'gan_final_2x_ft', 'gan_final_3x_ft',
    'derm_s040_2x_ft', 'derm_s005_2x_ft',
]

all_results = {}
for sc in COMPARE_SCENARIOS:
    run_dir = find_run_dir(sc)
    if run_dir and (run_dir / 'test_metrics.json').exists():
        m   = json.loads((run_dir / 'test_metrics.json').read_text())
        cfg = json.loads((run_dir / 'config.json').read_text()) if (run_dir / 'config.json').exists() else {}
        all_results[sc] = {'metrics': m, 'config': cfg}
    else:
        print(f'  {sc}: no completado')

if all_results:
    rows = []
    for sc, data in all_results.items():
        m, cfg = data['metrics'], data['config']
        rows.append({
            'Escenario':     sc,
            'Train mel':     cfg.get('train_mel', cfg.get('s1_train_mel', '?')),
            'AUC':           m['auc'],
            'Recall mel':    m['recall_mel'],
            'F1 mel':        m['f1_mel'],
            'Precision mel': m['precision_mel'],
            'Accuracy':      m['accuracy'],
        })
    df = pd.DataFrame(rows).set_index('Escenario')
    print(df.to_string())

    out = EXP_ROOT / 'comparison_finetune.json'
    out.write_text(json.dumps({sc: d['metrics'] for sc, d in all_results.items()}, indent=2))
    print(f'\nGuardado en {out}')

In [ ]:
if len(all_results) >= 2:
    ref    = {'real_only', 'gan64_2x', 'gan_final_2x', 'lora_2x'}
    ft_gan = {'lora_2x_ft', 'lora_3x_ft', 'gan_final_2x_ft', 'gan_final_3x_ft'}
    derm   = {'derm_s040_2x_ft', 'derm_s005_2x_ft'}
    def get_color(sc):
        if sc in ref:    return '#aec6e8'
        if sc in derm:   return '#8FBC8F'
        return '#C44E52'
    palette = {sc: get_color(sc) for sc in all_results}

    sc_list   = list(all_results.keys())
    metrics_k = [('auc', 'AUC'), ('recall_mel', 'Recall Melanoma'), ('f1_mel', 'F1 Melanoma')]

    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    for ax, (mk, title) in zip(axes, metrics_k):
        vals = [all_results[sc]['metrics'][mk] for sc in sc_list]
        cols = [palette[sc] for sc in sc_list]
        bars = ax.bar([s.replace('_', '\n') for s in sc_list], vals, color=cols)
        ax.set_ylim(0, 1.08); ax.set_title(title, fontsize=11)
        ax.tick_params(axis='x', labelsize=7)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=7)
        if 'real_only' in all_results:
            bl = all_results['real_only']['metrics'][mk]
            ax.axhline(bl, color='gray', linestyle='--', lw=0.8, label='real_only')
            ax.legend(fontsize=8)

    from matplotlib.patches import Patch
    fig.legend(handles=[
        Patch(facecolor='#aec6e8', label='Referencia (sin FT)'),
        Patch(facecolor='#C44E52', label='Fine-tuning 2 etapas — LoRA / GAN'),
        Patch(facecolor='#8FBC8F', label='Fine-tuning 2 etapas — Derm-T2IM'),
    ], loc='lower center', ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.08))
    fig.suptitle('HAM10000 — Efecto del fine-tuning de dos etapas', fontsize=13)
    fig.tight_layout()

    out = EXP_ROOT / 'comparison_finetune_plot.png'
    fig.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Plot guardado en {out}')